In [ ]:
! pip install ripser

In [4]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ripser import ripser
from scipy.signal import butter, filtfilt
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist
from sklearn.metrics import roc_auc_score


# ──────────────────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────────────────
PKL_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/pkl_data"
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/result_wesad_tpv_personalized_clean32"

FS_LABEL = 700
FS_BVP   = 64

WINDOW_SEC = 60
STEP_SEC   = 60   # non-overlap

USE_LABELS   = {1, 2}   # baseline, stress
STRESS_ID    = 2
MAJ_RATIO_TH = 0.95

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

SAVE_SUBJECT_BOXPLOTS = True
SAVE_POOLED_GRID_BOXPLOT = True

PALETTE = {
    "normal": "#1f77b4",   # blue
    "stress": "#d62728"    # red
}

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ──────────────────────────────────────────────────────────────────────────────
# FEATURE NAMES (32D features)
# ──────────────────────────────────────────────────────────────────────────────
TPV_FEATURE_NAMES = [
    "birth_mean",                  # 0
    "birth_std",                   # 1
    "death_mean",                  # 2
    "death_std",                   # 3
    "lifetime_mean",               # 4
    "lifetime_std",                # 5
    "lifetime_max",                # 6
    "lifetime_min",                # 7
    "lifetime_median",             # 8
    "lifetime_iqr",                # 9
    "lifetime_skew",               # 10
    "lifetime_kurtosis",           # 11
    "birth_max",                   # 12
    "birth_min",                   # 13
    "birth_median",                # 14
    "birth_skew",                  # 15
    "birth_kurtosis",              # 16
    "death_max",                   # 17
    "death_min",                   # 18
    "death_median",                # 19
    "death_skew",                  # 20
    "death_kurtosis",              # 21
    "num_lifetimes",               # 22
    "top1_lifetime",               # 23
    "top2_lifetime",               # 24
    "lifetime_max_div_min",        # 25
    "ph_entropy",                  # 26
    "betti_entropy",               # 27
    "avg_persistence_distance",    # 28
    "gini_index",                  # 29
    "lifetime_variance",           # 30
    "persistent_image_energy",     # 31
]

N_FEATURES = len(TPV_FEATURE_NAMES)


# ──────────────────────────────────────────────────────────────────────────────
# 0) Basic preprocessing
# ──────────────────────────────────────────────────────────────────────────────
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


# ──────────────────────────────────────────────────────────────────────────────
# 1) TPV (clean 32-d, no padding, no duplicate)
# ──────────────────────────────────────────────────────────────────────────────
def safe_skew(x: np.ndarray) -> float:
    if len(x) > 2 and np.std(x) > 1e-12:
        v = skew(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def safe_kurtosis(x: np.ndarray) -> float:
    if len(x) > 3 and np.std(x) > 1e-12:
        v = kurtosis(x)
        return float(v) if np.isfinite(v) else 0.0
    return 0.0


def extract_tpv_features(sig_1d: np.ndarray) -> np.ndarray:
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)

    if len(sig) < 3:
        return np.zeros(N_FEATURES, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(N_FEATURES, dtype=np.float32)

    # z-score normalize within segment
    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    # 2D delay embedding
    X = np.stack([sig[:-1], sig[1:]], axis=1)

    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]

    if H1.size == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births = births[mask]
    deaths = deaths[mask]
    lifetimes = lifetimes[mask]

    if len(lifetimes) == 0:
        return np.zeros(N_FEATURES, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)

    # additional topological metrics
    lifetime_sum = float(np.sum(lifetimes))
    lifetime_ratio = lifetimes / (lifetime_sum + 1e-8)
    ph_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin = float(np.min(births))
    bmax = float(np.max(births))
    if bmax - bmin < 1e-8:
        betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        betti_entropy = float(entropy(hist + 1e-10))

    avg_persistence_distance = (
        float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0
    )

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (
            2 * np.sum(np.arange(1, n + 1) * lifetimes_sorted)
        ) / (n * np.sum(lifetimes_sorted)) - (n + 1) / n
        gini_index = float(gini)
    else:
        gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))
    persistent_image_energy = float(np.sum(lifetimes ** 2))

    feats = [
        float(np.mean(births)),                                  # 0
        float(np.std(births)),                                   # 1
        float(np.mean(deaths)),                                  # 2
        float(np.std(deaths)),                                   # 3
        float(np.mean(lifetimes)),                               # 4
        float(np.std(lifetimes)),                                # 5
        float(np.max(lifetimes)),                                # 6
        float(np.min(lifetimes)),                                # 7
        float(np.median(lifetimes)),                             # 8
        float(iqr(lifetimes)),                                   # 9
        safe_skew(lifetimes),                                    # 10
        safe_kurtosis(lifetimes),                                # 11
        float(np.max(births)),                                   # 12
        float(np.min(births)),                                   # 13
        float(np.median(births)),                                # 14
        safe_skew(births),                                       # 15
        safe_kurtosis(births),                                   # 16
        float(np.max(deaths)),                                   # 17
        float(np.min(deaths)),                                   # 18
        float(np.median(deaths)),                                # 19
        safe_skew(deaths),                                       # 20
        safe_kurtosis(deaths),                                   # 21
        float(len(lifetimes)),                                   # 22
        float(lifetimes_sorted[-1]),                             # 23
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,  # 24
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),   # 25
        ph_entropy,                                              # 26
        betti_entropy,                                           # 27
        avg_persistence_distance,                                # 28
        gini_index,                                              # 29
        lifetime_variance,                                       # 30
        persistent_image_energy,                                 # 31
    ]

    feats = np.asarray(feats, dtype=np.float32)

    if len(feats) != N_FEATURES:
        raise ValueError(f"Expected {N_FEATURES} features, got {len(feats)}")

    return feats


# ──────────────────────────────────────────────────────────────────────────────
# 2) Build TPV table for one subject
# ──────────────────────────────────────────────────────────────────────────────
def build_bvp_tpv_table(
    pkl_path: str,
    subject_name: str,
    window_sec: int,
    step_sec: int,
    fs_label: int = 700,
    fs_bvp: int = 64,
    use_labels: set = {1, 2},
    stress_id: int = 2,
    maj_ratio_th: float = 0.95,
    verbose: bool = False,
) -> pd.DataFrame:

    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    bvp = np.asarray(s["signal"]["wrist"]["BVP"]).reshape(-1).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    dur_wrist = len(bvp) / fs_bvp
    labels = labels[: int(dur_wrist * fs_label)]

    bvp = fill_nan_with_median(bvp)
    bvp = bandpass_filter(bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)

    Wl = int(window_sec * fs_label)
    Sl = int(step_sec * fs_label)
    Wb = int(window_sec * fs_bvp)

    tpv_rows = []

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in use_labels:
            continue
        if maj_ratio < maj_ratio_th:
            continue

        t0 = start_l / fs_label
        start_b = int(round(t0 * fs_bvp))
        end_b = start_b + Wb

        if end_b > len(bvp):
            continue

        seg_bvp = bvp[start_b:end_b]
        if len(seg_bvp) != Wb:
            continue

        tpv = extract_tpv_features(seg_bvp)

        row = {
            "subject": subject_name,
            "pkl_path": pkl_path,
            "t_start_sec": float(t0),
            "t_end_sec": float(t0 + window_sec),
            "label_major": maj,
            "label_name": "stress" if maj == stress_id else "normal",
            "label_bin": 1 if maj == stress_id else 0,
            "major_ratio": maj_ratio,
        }

        for j, feat_name in enumerate(TPV_FEATURE_NAMES):
            row[f"BVP_TPV_{j}"] = float(tpv[j])
            row[f"BVP_TPV_NAME_{j}"] = feat_name

        tpv_rows.append(row)

        if verbose:
            print(f"[{subject_name}] KEEP t0={t0:.1f}s label_major={maj} ratio={maj_ratio:.3f}")

    return pd.DataFrame(tpv_rows)


# ──────────────────────────────────────────────────────────────────────────────
# 3) Build ALL subjects table
# ──────────────────────────────────────────────────────────────────────────────
def build_all_subjects_table(pkl_dir: str) -> pd.DataFrame:
    pkl_list = sorted(glob.glob(os.path.join(pkl_dir, "S*.pkl")))
    all_dfs = []

    print(f"[INFO] Found {len(pkl_list)} pkl files")

    for pkl_path in pkl_list:
        subject_name = os.path.splitext(os.path.basename(pkl_path))[0]
        print(f"\n[INFO] Processing {subject_name} ...")

        try:
            df_sub = build_bvp_tpv_table(
                pkl_path=pkl_path,
                subject_name=subject_name,
                window_sec=WINDOW_SEC,
                step_sec=STEP_SEC,
                fs_label=FS_LABEL,
                fs_bvp=FS_BVP,
                use_labels=USE_LABELS,
                stress_id=STRESS_ID,
                maj_ratio_th=MAJ_RATIO_TH,
                verbose=False,
            )

            print(f"[INFO] {subject_name}: extracted {len(df_sub)} windows")

            if len(df_sub) > 0:
                all_dfs.append(df_sub)

        except Exception as e:
            print(f"[WARN] Failed on {subject_name}: {e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    return pd.concat(all_dfs, axis=0, ignore_index=True)


# ──────────────────────────────────────────────────────────────────────────────
# Helper
# ──────────────────────────────────────────────────────────────────────────────
def get_feature_cols(df: pd.DataFrame):
    return [f"BVP_TPV_{i}" for i in range(N_FEATURES)]


def build_feature_definition_table() -> pd.DataFrame:
    return pd.DataFrame({
        "feature_index": list(range(N_FEATURES)),
        "feature_name": TPV_FEATURE_NAMES
    })


# ──────────────────────────────────────────────────────────────────────────────
# 4) Window-level summary
# ──────────────────────────────────────────────────────────────────────────────
def build_window_level_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    df_n = df[df["label_bin"] == 0]
    df_s = df[df["label_bin"] == 1]

    feature_cols = get_feature_cols(df)

    for i, col in enumerate(feature_cols):
        rows.append({
            "feature_index": i,
            "feature": col,
            "feature_name": TPV_FEATURE_NAMES[i],
            "normal_mean_window": df_n[col].mean(),
            "normal_std_window": df_n[col].std(),
            "normal_median_window": df_n[col].median(),
            "stress_mean_window": df_s[col].mean(),
            "stress_std_window": df_s[col].std(),
            "stress_median_window": df_s[col].median(),
            "mean_diff_window(stress-normal)": df_s[col].mean() - df_n[col].mean(),
            "median_diff_window(stress-normal)": df_s[col].median() - df_n[col].median(),
        })

    return pd.DataFrame(rows)


# ──────────────────────────────────────────────────────────────────────────────
# 5) Subject-level mean / diff summary
# ──────────────────────────────────────────────────────────────────────────────
def build_subject_level_summary(df: pd.DataFrame):
    feature_cols = get_feature_cols(df)

    subj_label_mean = (
        df.groupby(["subject", "label_name"])[feature_cols]
        .mean()
        .reset_index()
    )

    normal_df = subj_label_mean[subj_label_mean["label_name"] == "normal"].copy()
    stress_df = subj_label_mean[subj_label_mean["label_name"] == "stress"].copy()

    normal_df = normal_df.drop(columns=["label_name"]).set_index("subject")
    stress_df = stress_df.drop(columns=["label_name"]).set_index("subject")

    common_subjects = sorted(set(normal_df.index) & set(stress_df.index))
    normal_df = normal_df.loc[common_subjects]
    stress_df = stress_df.loc[common_subjects]

    rows = []
    for i, col in enumerate(feature_cols):
        diff = stress_df[col] - normal_df[col]

        rows.append({
            "feature_index": i,
            "feature": col,
            "feature_name": TPV_FEATURE_NAMES[i],
            "n_subjects": len(common_subjects),
            "normal_mean_subject": normal_df[col].mean(),
            "normal_std_subject": normal_df[col].std(),
            "normal_median_subject": normal_df[col].median(),
            "stress_mean_subject": stress_df[col].mean(),
            "stress_std_subject": stress_df[col].std(),
            "stress_median_subject": stress_df[col].median(),
            "mean_diff_subject(stress-normal)": diff.mean(),
            "std_diff_subject(stress-normal)": diff.std(),
            "median_diff_subject(stress-normal)": diff.median(),
        })

    summary_df = pd.DataFrame(rows)

    diff_df = (stress_df - normal_df).reset_index()
    diff_df.insert(1, "comparison", "stress-normal")

    return summary_df, subj_label_mean, diff_df


# ──────────────────────────────────────────────────────────────────────────────
# 6) Subject-wise personalized metrics
# ──────────────────────────────────────────────────────────────────────────────
def compute_subject_feature_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each subject and each feature:
    - normal_mean / stress_mean
    - diff = stress - normal
    - direction
    - Cohen's d
    - ROC AUC
    """
    feature_cols = get_feature_cols(df)
    subjects = sorted(df["subject"].unique())

    rows = []

    for subj in subjects:
        df_sub = df[df["subject"] == subj].copy()

        df_n = df_sub[df_sub["label_bin"] == 0]
        df_s = df_sub[df_sub["label_bin"] == 1]

        # need both classes
        if len(df_n) == 0 or len(df_s) == 0:
            continue

        for i, col in enumerate(feature_cols):
            normal_vals = df_n[col].dropna().values.astype(float)
            stress_vals = df_s[col].dropna().values.astype(float)

            if len(normal_vals) == 0 or len(stress_vals) == 0:
                continue

            normal_mean = float(np.mean(normal_vals))
            stress_mean = float(np.mean(stress_vals))
            normal_std = float(np.std(normal_vals, ddof=1)) if len(normal_vals) > 1 else 0.0
            stress_std = float(np.std(stress_vals, ddof=1)) if len(stress_vals) > 1 else 0.0

            diff = stress_mean - normal_mean

            if diff > 0:
                direction = "increase"
            elif diff < 0:
                direction = "decrease"
            else:
                direction = "no_change"

            pooled_std = np.sqrt((normal_std ** 2 + stress_std ** 2) / 2.0)
            if pooled_std < 1e-12:
                cohen_d = 0.0
            else:
                cohen_d = diff / pooled_std

            # ROC AUC
            y_true = np.concatenate([
                np.zeros(len(normal_vals), dtype=int),
                np.ones(len(stress_vals), dtype=int)
            ])
            scores = np.concatenate([normal_vals, stress_vals])

            try:
                auc_raw = float(roc_auc_score(y_true, scores))
            except Exception:
                auc_raw = np.nan

            # direction-independent separability
            if np.isnan(auc_raw):
                auc_best = np.nan
            else:
                auc_best = max(auc_raw, 1.0 - auc_raw)

            rows.append({
                "subject": subj,
                "feature_index": i,
                "feature": col,
                "feature_name": TPV_FEATURE_NAMES[i],
                "n_normal_windows": len(normal_vals),
                "n_stress_windows": len(stress_vals),
                "normal_mean": normal_mean,
                "normal_std": normal_std,
                "stress_mean": stress_mean,
                "stress_std": stress_std,
                "diff_stress_minus_normal": diff,
                "direction": direction,
                "cohen_d": cohen_d,
                "roc_auc_raw": auc_raw,
                "roc_auc_best": auc_best,
            })

    return pd.DataFrame(rows)


def compute_feature_consistency_summary(subject_feature_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate subject-wise results into feature-wise summary:
    - increase_count / decrease_count / no_change_count
    - consistency ratio
    - mean/std of Cohen's d
    - mean/std of AUC
    """
    rows = []

    for i in range(N_FEATURES):
        df_f = subject_feature_metrics[subject_feature_metrics["feature_index"] == i].copy()
        if len(df_f) == 0:
            continue

        increase_count = int((df_f["direction"] == "increase").sum())
        decrease_count = int((df_f["direction"] == "decrease").sum())
        no_change_count = int((df_f["direction"] == "no_change").sum())
        n_subjects = len(df_f)

        consistency_ratio = max(increase_count, decrease_count) / n_subjects if n_subjects > 0 else np.nan
        dominant_direction = (
            "increase" if increase_count > decrease_count
            else "decrease" if decrease_count > increase_count
            else "tie"
        )

        rows.append({
            "feature_index": i,
            "feature": f"BVP_TPV_{i}",
            "feature_name": TPV_FEATURE_NAMES[i],
            "n_subjects": n_subjects,
            "increase_count": increase_count,
            "decrease_count": decrease_count,
            "no_change_count": no_change_count,
            "dominant_direction": dominant_direction,
            "consistency_ratio": consistency_ratio,
            "mean_diff_stress_minus_normal": df_f["diff_stress_minus_normal"].mean(),
            "std_diff_stress_minus_normal": df_f["diff_stress_minus_normal"].std(),
            "mean_cohen_d": df_f["cohen_d"].mean(),
            "std_cohen_d": df_f["cohen_d"].std(),
            "mean_roc_auc_raw": df_f["roc_auc_raw"].mean(),
            "std_roc_auc_raw": df_f["roc_auc_raw"].std(),
            "mean_roc_auc_best": df_f["roc_auc_best"].mean(),
            "std_roc_auc_best": df_f["roc_auc_best"].std(),
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(
        by=["consistency_ratio", "mean_roc_auc_best", "mean_cohen_d"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    return out


# ──────────────────────────────────────────────────────────────────────────────
# 7) Boxplots
# ──────────────────────────────────────────────────────────────────────────────
def plot_grid_boxplots(df: pd.DataFrame, save_path: str, title_prefix: str = "", n_cols: int = 4):
    feature_cols = get_feature_cols(df)
    n_feats = len(feature_cols)
    n_rows = int(np.ceil(n_feats / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.5 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(feature_cols):
        ax = axes[i]

        sns.boxplot(
            data=df,
            x="label_name",
            y=col,
            hue="label_name",
            order=["normal", "stress"],
            palette=PALETTE,
            dodge=False,
            legend=False,
            ax=ax
        )

        ax.set_title(f"{col}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

    for j in range(n_feats, len(axes)):
        axes[j].axis("off")

    if title_prefix:
        fig.suptitle(title_prefix, fontsize=14, y=1.01)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()


def save_subject_boxplots(df_all: pd.DataFrame, output_dir: str):
    subject_plot_dir = os.path.join(output_dir, "subject_boxplots_clean32")
    os.makedirs(subject_plot_dir, exist_ok=True)

    subjects = sorted(df_all["subject"].unique())
    for subj in subjects:
        df_sub = df_all[df_all["subject"] == subj].copy()
        if len(df_sub) == 0:
            continue

        save_path = os.path.join(subject_plot_dir, f"{subj}_tpv_clean32_boxplot_grid.png")
        plot_grid_boxplots(
            df=df_sub,
            save_path=save_path,
            title_prefix=f"{subj}: normal vs stress",
            n_cols=4
        )
        print(f"[INFO] Saved subject boxplot grid -> {save_path}")


# ──────────────────────────────────────────────────────────────────────────────
# 8) Long-format raw TPV save (for future vizualize)
# ──────────────────────────────────────────────────────────────────────────────
def build_long_format_raw_tpv(df_all: pd.DataFrame) -> pd.DataFrame:
    """
    Converts wide window-level TPV table to long format:
    one row per (window, feature)
    """
    id_cols = [
        "subject", "pkl_path", "t_start_sec", "t_end_sec",
        "label_major", "label_name", "label_bin", "major_ratio"
    ]
    feature_cols = get_feature_cols(df_all)

    df_long = df_all.melt(
        id_vars=id_cols,
        value_vars=feature_cols,
        var_name="feature",
        value_name="value"
    )

    feature_map = {f"BVP_TPV_{i}": TPV_FEATURE_NAMES[i] for i in range(N_FEATURES)}
    index_map = {f"BVP_TPV_{i}": i for i in range(N_FEATURES)}

    df_long["feature_index"] = df_long["feature"].map(index_map)
    df_long["feature_name"] = df_long["feature"].map(feature_map)

    # reorder
    df_long = df_long[
        id_cols + ["feature_index", "feature", "feature_name", "value"]
    ].copy()

    return df_long


# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"[INFO] Using clean TPV feature set with {N_FEATURES} valid features")
    print(build_feature_definition_table())

    # ─────────────────────────
    # A) Build raw TPV window-level master table
    # ─────────────────────────
    df_all = build_all_subjects_table(PKL_DIR)

    print("\n[INFO] final shape:", df_all.shape)

    if len(df_all) == 0:
        print("[WARN] No valid windows were extracted from any subject.")
        raise SystemExit

    # ─────────────────────────
    # B) Save feature definition
    # ─────────────────────────
    feat_def_csv = os.path.join(OUTPUT_DIR, "WESAD_tpv_feature_definition_clean32.csv")
    build_feature_definition_table().to_csv(feat_def_csv, index=False)
    print(f"[INFO] Saved feature definition: {feat_def_csv}")

    # ─────────────────────────
    # C) Save MASTER raw window-level dataset
    # ─────────────────────────
    main_csv = os.path.join(OUTPUT_DIR, "WESAD_all_subjects_wrist_bvp_tpv_clean32_baseline_vs_stress.csv")
    df_all.to_csv(main_csv, index=False)
    print(f"[INFO] Saved main raw window-level CSV: {main_csv}")

    # ─────────────────────────
    # D) Save LONG-format raw dataset
    # ─────────────────────────
    long_csv = os.path.join(OUTPUT_DIR, "WESAD_all_subjects_wrist_bvp_tpv_clean32_long_format.csv")
    df_long = build_long_format_raw_tpv(df_all)
    df_long.to_csv(long_csv, index=False)
    print(f"[INFO] Saved long-format raw CSV: {long_csv}")

    # ─────────────────────────
    # E) Window-level summary
    # ─────────────────────────
    window_summary = build_window_level_summary(df_all)
    window_summary_csv = os.path.join(OUTPUT_DIR, "WESAD_window_level_summary_clean32.csv")
    window_summary.to_csv(window_summary_csv, index=False)
    print(f"[INFO] Saved window-level summary: {window_summary_csv}")

    # ─────────────────────────
    # F) Subject-level mean / diff summary
    # ─────────────────────────
    subject_summary, subj_label_mean, subj_diff = build_subject_level_summary(df_all)

    subject_summary_csv = os.path.join(OUTPUT_DIR, "WESAD_subject_level_summary_clean32.csv")
    subject_summary.to_csv(subject_summary_csv, index=False)
    print(f"[INFO] Saved subject-level summary: {subject_summary_csv}")

    subj_mean_csv = os.path.join(OUTPUT_DIR, "WESAD_subject_labelwise_feature_means_clean32.csv")
    subj_label_mean.to_csv(subj_mean_csv, index=False)
    print(f"[INFO] Saved subject-label feature means: {subj_mean_csv}")

    subj_diff_csv = os.path.join(OUTPUT_DIR, "WESAD_subject_feature_diff_stress_minus_normal_clean32.csv")
    subj_diff.to_csv(subj_diff_csv, index=False)
    print(f"[INFO] Saved subject-wise feature diff CSV: {subj_diff_csv}")

    # ─────────────────────────
    # G) Subject-feature detailed metrics
    #    (direction, Cohen's d, ROC AUC for each subject x feature)
    # ─────────────────────────
    subject_feature_metrics = compute_subject_feature_metrics(df_all)
    subject_feature_metrics_csv = os.path.join(
        OUTPUT_DIR,
        "WESAD_subject_feature_metrics_clean32_direction_cohend_auc.csv"
    )
    subject_feature_metrics.to_csv(subject_feature_metrics_csv, index=False)
    print(f"[INFO] Saved subject-feature detailed metrics: {subject_feature_metrics_csv}")

    # ─────────────────────────
    # H) Feature-level consistency summary
    # ─────────────────────────
    feature_consistency_summary = compute_feature_consistency_summary(subject_feature_metrics)
    consistency_csv = os.path.join(
        OUTPUT_DIR,
        "WESAD_feature_consistency_cohend_auc_summary_clean32.csv"
    )
    feature_consistency_summary.to_csv(consistency_csv, index=False)
    print(f"[INFO] Saved feature consistency summary: {consistency_csv}")

    # ─────────────────────────
    # I) Subject / label counts
    # ─────────────────────────
    print("\n[INFO] Window-level label counts")
    print(df_all["label_name"].value_counts())

    print("\n[INFO] Subject counts")
    print(df_all.groupby(["subject", "label_name"]).size().unstack(fill_value=0))

    # ─────────────────────────
    # J) Pooled boxplot grid
    # ─────────────────────────
    if SAVE_POOLED_GRID_BOXPLOT:
        pooled_grid_path = os.path.join(
            OUTPUT_DIR,
            "WESAD_all_subjects_tpv_clean32_boxplot_grid_blue_red.png"
        )
        plot_grid_boxplots(df_all, pooled_grid_path, title_prefix="All Subjects (Pooled)", n_cols=4)
        print(f"[INFO] Saved pooled grid boxplot -> {pooled_grid_path}")

    # ─────────────────────────
    # K) Subject-wise boxplots
    # ─────────────────────────
    if SAVE_SUBJECT_BOXPLOTS:
        save_subject_boxplots(df_all, OUTPUT_DIR)

    # ─────────────────────────
    # L) Quick prints
    # ─────────────────────────
    top_by_consistency = feature_consistency_summary[
        ["feature_index", "feature_name", "increase_count", "decrease_count",
         "dominant_direction", "consistency_ratio", "mean_cohen_d", "mean_roc_auc_best"]
    ].head(10)

    print("\n[INFO] Top 10 features by consistency / AUC / Cohen's d")
    print(top_by_consistency.to_string(index=False))

    print("\n[INFO] Example: subject-wise detailed metrics (first 20 rows)")
    print(subject_feature_metrics.head(20).to_string(index=False))

[INFO] Using clean TPV feature set with 32 valid features
    feature_index              feature_name
0               0                birth_mean
1               1                 birth_std
2               2                death_mean
3               3                 death_std
4               4             lifetime_mean
5               5              lifetime_std
6               6              lifetime_max
7               7              lifetime_min
8               8           lifetime_median
9               9              lifetime_iqr
10             10             lifetime_skew
11             11         lifetime_kurtosis
12             12                 birth_max
13             13                 birth_min
14             14              birth_median
15             15                birth_skew
16             16            birth_kurtosis
17             17                 death_max
18             18                 death_min
19             19              death_median
20             20 